# DSFB Phosphoric Colab Reproducibility Run

This notebook starts from a fresh Colab runtime, clones the repository, installs QEMU/OVMF dependencies, runs the active DSFB court verification gate, and displays the v0.3 DSFB demo forensic evidence outputs.

Claim scope: a passing run means this Colab runtime, at the resolved git commit and package versions printed below, passed the checked commands. It is not a claim of bit-identical behavior across every host, QEMU, OVMF, filesystem-tool, or kernel version.

## What, Why, and How

This notebook is the clean external smoke path for the DSFB Phosphoric v0.3 court gate. It starts from a fresh Colab runtime, installs the host tools, clones the public repository, runs `make -k verify-court-active`, and then prints/downloads the evidence files that explain what was checked.

The core DSFB model is a deterministic residual pipeline:

```text
(y_hat, y, phi, s) -> r -> (d, sigma) -> E -> g -> tau -> C
```

Here `y_hat` is predicted or declared state, `y` is observed state, `phi` is phase/context, `s` is source or authority context, `r` is a typed residual, `(d, sigma)` is drift/slew classification, `E` is the admissibility envelope, `g` is grammar or motif state, `tau` is trust state, and `C` is the byte-deterministic certificate or verdict. The v0.3 notebook checks a narrow executable slice of that program: boot the demo, emit typed residuals, package them into PFI0 evidence, replay verdict fixtures, and verify the court gates.

Each residual record is 32 bytes: `kind:u8 | arch_id:u8 | seq:u16 | cycle:u64 | payload:[u8;14] | chain_hash:[u8;4] | pad:[u8;2]`. The 4-byte `chain_hash` is a deterministic fixture-continuity mixer, not a cryptographic hash. The verifier uses primes `[31, 131, 524287, 16777213]` and computes `chain_hash[n] = (prev[n] + sum(event[k] * primes[n])) mod 256` over a 28-byte event vector. SHA-256 is used separately for artifact and stream hashes.

PFI0 is the replayable case-file container. The layout gate checks the `PFI0` magic, residual count, reserved zero regions, SHA-256 stream hash, closed residual kind taxonomy, monotonic sequence numbers, chain continuity, and final-chain footer anchoring. The R5 MMIO boundary fixture pins `declared=0x1000..0x10FF`, `observed=0x1100`, and `chain_hash=8aa2ca5e`.

The code path to keep in mind is: `apps/dsfb_demo/` -> `tools/phosphoric/write_dsfb_efi.sh` -> `tools/qemu-run/run_dsfb_demo.sh` -> `build/uefi-demo/dsfb/dsfb_demo.pfi` -> `tools/verify/fixtures/verdicts/`. The command `make -k verify-court-active` ties that QEMU runtime evidence to the active forensic checks. `[scaffold]` lines are disclosed limitations for historical or out-of-tree checks that are absent from a public source-only clone; they are not counted as empirical pass evidence.

In [ ]:
import os

REPO_URL = os.environ.get("DSFB_REPO_URL", "https://github.com/infinityabundance/dsfb.git")
REF = os.environ.get("DSFB_REF", "main")
WORKDIR = os.environ.get("DSFB_WORKDIR", "/content/dsfb-repro")
PROJECT = f"{WORKDIR}/crates/dsfb-phosphoric"

os.environ.update({
    "REPO_URL": REPO_URL,
    "REF": REF,
    "WORKDIR": WORKDIR,
    "PROJECT": PROJECT,
})

print(f"REPO_URL={REPO_URL}")
print(f"REF={REF}")
print(f"WORKDIR={WORKDIR}")
print(f"PROJECT={PROJECT}")

## 1. Install host dependencies

Colab runtimes are ephemeral. This cell installs the host tools required by the QEMU/OVMF verification path.

In [ ]:
%%bash
set -euxo pipefail
sudo apt-get update
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends \
  git make coreutils gawk dosfstools mtools qemu-system-x86 ovmf

## 2. Fresh clone

The clone directory is removed first so each notebook run starts from repository source, not from previous Colab state.

In [ ]:
%%bash
set -euxo pipefail
rm -rf "$WORKDIR"
git clone "$REPO_URL" "$WORKDIR"
cd "$WORKDIR"
git checkout "$REF"
git rev-parse HEAD | tee /content/dsfb_colab_commit.txt
test -d "$PROJECT"
find "$PROJECT" -maxdepth 1 -type f -name README.md -print

## 3. Environment evidence

This records the commit, QEMU version, OVMF files, and package versions used by this run.

In [ ]:
%%bash
set -euxo pipefail
cd "$PROJECT"
{
  echo "repo_url=$(git remote get-url origin)"
  echo "commit=$(git rev-parse HEAD)"
  echo "ref=$REF"
  echo
  echo "## qemu-system-x86_64 --version"
  qemu-system-x86_64 --version | sed -n '1,3p'
  echo
  echo "## awk --version"
  awk --version | sed -n '1,3p'
  echo
  echo "## apt package versions"
  dpkg-query -W -f='${binary:Package}\t${Version}\n' \
    git make coreutils gawk dosfstools mtools qemu-system-x86 ovmf || true
  echo
  echo "## OVMF firmware candidates"
  find /usr/share/OVMF /usr/share/edk2 -type f -name '*.fd' -print 2>/dev/null | sort || true
} | tee /content/dsfb-environment.txt

## 4. Run the active verification gate

The verification command is `make -k verify-court-active`. Output is saved to `/content/verify-court-active.log` and the raw shell status is saved to `/content/verify-court-active.status`. Public source-only runs may print `[scaffold]` disclosures for historical or out-of-tree producer cross-checks that are intentionally absent from the clean clone; those disclosures are collected separately and are not counted as empirical pass evidence.

In [ ]:
%%bash
set -uxo pipefail
cd "$PROJECT"
set +e
make -k verify-court-active 2>&1 | tee /content/verify-court-active.log
pipe_status=("${PIPESTATUS[@]}")
make_rc="${pipe_status[0]}"
tee_rc="${pipe_status[1]}"
set -e
printf 'make_rc=%s\ntee_rc=%s\n' "$make_rc" "$tee_rc" | tee /content/verify-court-active.status
exit 0


In [ ]:
%%bash
set -euo pipefail
cat /content/verify-court-active.status
if grep -E '(^make: \*\*\* .*(Error|Stop)|Error [0-9]+|FAIL:|No such file or directory)' /content/verify-court-active.log; then
  echo '[colab-gate] FAIL - concrete error marker found in verification log' >&2
  exit 1
fi
make_rc="$(awk -F= '$1 == "make_rc" { print $2 }' /content/verify-court-active.status)"
if [ "${make_rc:-unknown}" != "0" ]; then
  echo "[colab-gate] NOTE - raw make exited $make_rc; no concrete failure marker found, so this is treated as make -k/scaffold status under logged evidence"
fi
grep -E '^\[scaffold\]' /content/verify-court-active.log | tee /content/dsfb-scaffold-disclosures.txt || true
echo '[colab-gate] OK - verify-court-active log triage passed; scaffold disclosures, if any, were recorded'


## 5. Collect v0.3 DSFB demo evidence

This creates `/content/dsfb-evidence.txt` with the DSFB QEMU markers, linked artifact manifest, verdict text, key hashes, and scaffold disclosures from the `verify-court-active` run.

In [ ]:
%%bash
set -euxo pipefail
cd "$PROJECT"
{
  echo "# DSFB Colab evidence"
  date -u '+generated_utc=%Y-%m-%dT%H:%M:%SZ'
  echo "repo_url=$(git remote get-url origin)"
  echo "commit=$(git rev-parse HEAD)"
  echo
  echo "## scaffold disclosures"
  cat /content/dsfb-scaffold-disclosures.txt 2>/dev/null || true
  echo
  echo "## qemu-debug.log"
  sed -n '1,120p' build/uefi-demo/dsfb/qemu-debug.log
  echo
  echo "## linked-artifact.txt"
  sed -n '1,140p' build/uefi-demo/dsfb/linked-artifact.txt
  echo
  echo "## verdict: dsfb_demo.expect"
  cat tools/verify/fixtures/verdicts/dsfb_demo.expect
  echo
  echo "## verdict: mmio_boundary_violation.expect"
  cat tools/verify/fixtures/verdicts/mmio_boundary_violation.expect
  echo
  echo "## sha256"
  sha256sum \
    build/uefi-demo/dsfb/esp/EFI/BOOT/BOOTX64.EFI \
    build/uefi-demo/dsfb/dsfb_demo.pfi \
    tests/golden/dsfb_demo.pfi \
    tools/verify/fixtures/verdicts/dsfb_demo.expect \
    tools/verify/fixtures/verdicts/mmio_boundary_violation.expect
} | tee /content/dsfb-evidence.txt

## 6. Display evidence files

In [ ]:
from pathlib import Path
import os

project = Path(os.environ["PROJECT"])
paths = [
    Path("/content/dsfb-environment.txt"),
    Path("/content/verify-court-active.log"),
    Path("/content/verify-court-active.status"),
    Path("/content/dsfb-scaffold-disclosures.txt"),
    project / "build/uefi-demo/dsfb/qemu-debug.log",
    project / "build/uefi-demo/dsfb/linked-artifact.txt",
    project / "tools/verify/fixtures/verdicts/dsfb_demo.expect",
    project / "tools/verify/fixtures/verdicts/mmio_boundary_violation.expect",
    Path("/content/dsfb-evidence.txt"),
]

for path in paths:
    print(f"\n===== {path} =====")
    text = path.read_text(errors="replace")
    print(text[:20000])
    if len(text) > 20000:
        print("\n[truncated for notebook display]")

## 7. Download review artifacts

This cell is Colab-specific. It downloads the verification log, raw status, environment record, and v0.3 DSFB demo evidence record.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import files
except Exception as exc:
    print(f"google.colab download helper is unavailable in this runtime: {exc}")
else:
    project = Path(os.environ["PROJECT"])
    download_paths = [
        "/content/verify-court-active.log",
        "/content/verify-court-active.status",
        "/content/dsfb-environment.txt",
        "/content/dsfb-evidence.txt",
    ]
    for path in download_paths:
        print(f"Downloading {path}")
        files.download(path)

## Repository and IP Notice

Repository: [https://github.com/infinityabundance/dsfb/tree/main/crates/dsfb-phosphoric](https://github.com/infinityabundance/dsfb/tree/main/crates/dsfb-phosphoric)

Licensed under Apache 2.0 - Copyright 2026 - Invariant Forge LLC. Commercial use requires a separate license. licensing@invariantforge.net